# Safety Stock Assignments
A notebook summarizing both Assignment 1 and Assignment 2.
### Packages and Data

In [36]:
import pandas as pd
import numpy as np
from scipy import stats

trans = pd.read_csv("../Data/transaction_data.csv")
meta = pd.read_csv("../Data/meta_data.csv")

## Data Cleaning
### Column Renaming

Before we do any work with the data, we have to make sure it is cleaned first. First, we start with using the metadata as a guide to ensure a consistant naming convention.

In [37]:
meta

,feature,description
0,sku_number,Eight character stock keeping unit number
1,inventory_type,Types of inventory including finished good - w...
2,stocking_type,Stockign type policy including make to stock -...
3,lead_time,Total lead time in days
4,unit_price,Unit price US dollar value
5,manufacturing_site,Manufacturing site ID starting with country co...
6,division_code,Four character business division ID
7,transaction_date,Date of Transaction in Year-Month-Day format
8,order_quantity,Order quantity - rounded up for convenience


In [38]:
trans

,sku_number,inventory_type,stocking_type,leadTime,unit_price,Manufacturing Site,division Code,transaction_date,order_quantity
0,796AE627,WIP,MTO,14,46.540779,US-F4,B17E,2023-01-01,18
1,7F93DE39,RM,MTS,28,1003.940102,US-F4,37AC,2023-01-01,103
2,F7353AF9,FG,MTS,21,195.603667,JP-82,E9E0,2023-01-01,-6
3,DF10CC32,FG,MTS,14,50.546061,US-F4,615C,2023-01-01,2
4,F80559F8,RM,MTO,28,997.265710,US-33,E9E0,2023-01-01,120
...,...,...,...,...,...,...,...,...,...
772802,8D256BFE,RM,MTS,28,1005.262274,NaN,C7DB,2025-12-30,89
772803,7BC08068,FG,MTO,28,1006.640914,US-F4,8552,2025-12-30,75
772804,DB52452A,FG,MTS,28,1001.112377,US-F4,B23B,2025-12-30,118
772805,6EDF0167,FG,MTO,28,1009.120709,JP-82,NaN,2025-12-30,52


In [39]:
transa = trans.rename(columns={"leadTime":"lead_time",
                      "Manufacturing Site":"manufacturing_site", 
                      "division Code":"division_code"})
transa

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
0,796AE627,WIP,MTO,14,46.540779,US-F4,B17E,2023-01-01,18
1,7F93DE39,RM,MTS,28,1003.940102,US-F4,37AC,2023-01-01,103
2,F7353AF9,FG,MTS,21,195.603667,JP-82,E9E0,2023-01-01,-6
3,DF10CC32,FG,MTS,14,50.546061,US-F4,615C,2023-01-01,2
4,F80559F8,RM,MTO,28,997.265710,US-33,E9E0,2023-01-01,120
...,...,...,...,...,...,...,...,...,...
772802,8D256BFE,RM,MTS,28,1005.262274,NaN,C7DB,2025-12-30,89
772803,7BC08068,FG,MTO,28,1006.640914,US-F4,8552,2025-12-30,75
772804,DB52452A,FG,MTS,28,1001.112377,US-F4,B23B,2025-12-30,118
772805,6EDF0167,FG,MTO,28,1009.120709,JP-82,NaN,2025-12-30,52


### Anomalous Data
From there, we can take an overall look at the data to see if there are any anomalies.

In [40]:
transa.describe()

,lead_time,unit_price,order_quantity
count,772807.000000,772807.000000,772807.000000
mean,25.768879,798.432806,79.796756
std,10.508581,375.278660,41.964340
min,14.000000,-1005.424302,-124.000000
25%,28.000000,980.444049,63.000000
50%,28.000000,996.593102,92.000000
75%,28.000000,1005.424302,109.000000
max,2800.000000,1028.705768,193.000000


Though everything looks reasonable at first, there are some glaring issues that need to be addressed. Firstly, the order quantity and unit price values can't be negative, as the data set does not include information about returns. So the first step is to convert those negative values into positive values.

In [41]:
transa["unit_price"] = np.where(transa["unit_price"]<0, transa["unit_price"]*-1, transa["unit_price"])
transa["order_quantity"] = np.where(transa["order_quantity"]<0, transa["order_quantity"]*-1, transa["order_quantity"])

The other issue is that the lead time column shows an absurdly high value. When investigating the issue, it appears to be the the expected lead time values multiplied by a factor of 100. Removing the appended "00" form these values will bring them back into a resonable range.

In [42]:
transa[transa["lead_time"] > 28]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
13043,22C4EED0,RM,MTO,2800,990.481131,JP-80,E9E0,2023-01-19,95
160964,CB02C765,RM,MTS,1400,50.690146,PL-8A,F06D,2023-08-16,4
325684,09774D6A,RM,MTS,2800,112.920915,US-42,0DFF,2024-04-06,3
422755,3846D0DD,RM,MTS,2800,1016.790699,JP-82,74D4,2024-08-21,133
473337,E802533E,FG,MTS,2100,199.915359,JP-4F,9667,2024-11-01,24
556496,943929D9,FG,MTO,2800,975.133388,JP-4F,42BB,2025-02-27,116
618609,E3669638,FG,ATO,2800,1000.412434,JP-82,6A6F,2025-05-26,126
688596,4D953270,FG,MTS,2800,980.444049,JP-4F,8552,2025-09-02,106
695541,222DA157,FG,NaN,2800,1006.036826,JP-57,B17E,2025-09-12,92
704481,1D670AF7,FG,MTS,2800,1005.065868,JP-4F,B23B,2025-09-25,124


In [43]:
transa = transa.replace({1400:14, 2100:21, 2800:28})

In [44]:
transa.describe()

,lead_time,unit_price,order_quantity
count,772807.00000,772807.000000,772807.000000
mean,25.73570,798.451860,80.581633
std,4.76934,375.238118,40.436722
min,14.00000,35.310439,0.000000
25%,28.00000,980.444049,63.000000
50%,28.00000,996.593102,92.000000
75%,28.00000,1005.424302,109.000000
max,28.00000,1028.705768,193.000000


### Missing Data
With the known data values taken care of, we now need to see if there are any missing data values in the dataset.

In [45]:
transa.isna().sum()

sku_number              2232
inventory_type         17557
stocking_type          16383
lead_time                  0
unit_price                 0
manufacturing_site    149524
division_code          53773
transaction_date           0
order_quantity             0
dtype: int64

From this summarization, we can see that there are missing data in 5 columns, all categorical data. With the varying amounts, it would be destructive to any future analyses if we were to just remove them from the data set.

To compensate, we'll replace the missing values with an "Unknown" value, allowing us to retain the order quantity, lead time, and unit price data.

In [46]:
transa = transa.replace(np.nan, "Unknown")

In [47]:
transa.isna().sum()

sku_number            0
inventory_type        0
stocking_type         0
lead_time             0
unit_price            0
manufacturing_site    0
division_code         0
transaction_date      0
order_quantity        0
dtype: int64

## Data Transformation
### Country Breakdown
From there, we can start working with our data set. For broader analyses, we can add a country column to more easily breakdown where each SKU is manufactured. We can extract this information from the manufacturing site data, as each value has a country code that can be compared against the meta data to determine the country of manufacturing.

In [48]:
transa["manufacturing_site"].unique()

<StringArray>
[  'US-F4',   'JP-82',   'US-33',   'JP-4F', 'Unknown',   'US-24',   'US-42',
   'JP-80',   'US-30',   'US-DA',   'JP-57',   'CA-BB',   'PL-6F',   'US-4E',
   'PL-42',   'PL-8A',   'CA-23',   'JP-93',   'US-FC']
Length: 19, dtype: str

In [49]:
transa.insert(5, "country", 
                    np.where(transa["manufacturing_site"].str.contains("US"), "United States",
                    np.where(transa["manufacturing_site"].str.contains("JP"), "Japan",
                    np.where(transa["manufacturing_site"].str.contains("CA"), "Canada", 
                             "Poland"))))

In [50]:
transa.sample(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,country,manufacturing_site,division_code,transaction_date,order_quantity
352818,AA745368,WIP,MTO,28,995.306099,Japan,JP-82,9667,2024-05-14,87
93185,F61BDBAA,RM,MTO,28,1000.169659,Poland,Unknown,7DD3,2023-05-12,94
192666,77CB8611,WIP,MTS,28,1012.341272,Japan,JP-82,89DA,2023-10-01,95
757430,081AE6AD,WIP,MTS,28,998.336922,Poland,Unknown,B23B,2025-12-09,53
140652,20F7AA6B,FG,MTS,28,1003.275627,United States,US-F4,9667,2023-07-18,133
666396,280700E7,FG,MTS,28,1000.212000,United States,US-F4,Unknown,2025-08-02,93
595593,F41CF5ED,WIP,MTO,28,1010.432510,Japan,JP-82,2C03,2025-04-23,92
630433,C9461F48,FG,MTS,28,995.736331,Japan,JP-4F,2458,2025-06-12,104
380155,D3FF8512,FG,Unknown,28,988.555222,Japan,JP-4F,3BC4,2024-06-22,95
568646,26C6B984,FG,MTS,28,96.940032,United States,US-F4,B17E,2025-03-16,8


With the new column, we can now filter the data so that we are working only with made-to-stock (MTS) SKUs, which is the subset we want to focus on.

In [51]:
trans_mts = transa.query('stocking_type == "MTS"')
trans_mts

,sku_number,inventory_type,stocking_type,lead_time,unit_price,country,manufacturing_site,division_code,transaction_date,order_quantity
1,7F93DE39,RM,MTS,28,1003.940102,United States,US-F4,37AC,2023-01-01,103
2,F7353AF9,FG,MTS,21,195.603667,Japan,JP-82,E9E0,2023-01-01,6
3,DF10CC32,FG,MTS,14,50.546061,United States,US-F4,615C,2023-01-01,2
5,AAFE4414,FG,MTS,28,1004.061329,Japan,JP-82,3B50,2023-01-01,112
6,1D7F004E,FG,MTS,28,994.476636,Japan,JP-4F,6F69,2023-01-01,80
...,...,...,...,...,...,...,...,...,...,...
772799,20F7AA6B,FG,MTS,28,1003.275627,Japan,JP-82,6A6F,2025-12-30,97
772800,FDC8BB25,RM,MTS,14,42.682104,Japan,JP-82,2C03,2025-12-30,14
772801,280700E7,FG,MTS,28,1000.212000,Poland,Unknown,36FB,2025-12-30,97
772802,8D256BFE,RM,MTS,28,1005.262274,Poland,Unknown,C7DB,2025-12-30,89


### Initial Look
Before continuing to the safety stock analysis, we can take a look at the data we have for dimensional information. For instance, we can look to see how many unique SKUs, manufacturing sites, and divisions are in the data set.

In [52]:
pd.DataFrame({"unique_sku" : trans_mts["sku_number"].nunique(),
              "unique_manu" : trans_mts["manufacturing_site"].nunique(),
              "unique_division" : trans_mts["division_code"].nunique()}, index= [0])

,unique_sku,unique_manu,unique_division
0,506,19,55


We can also look at the top and bottom 10 transactions by order quantity.

In [53]:
top_orders=trans_mts.sort_values("order_quantity", ascending=False).head(10)
bottom_orders=trans_mts.sort_values("order_quantity", ascending=False).tail(10)

pd.concat([top_orders, bottom_orders])

,sku_number,inventory_type,stocking_type,lead_time,unit_price,country,manufacturing_site,division_code,transaction_date,order_quantity
16266,01DD6D59,WIP,MTS,28,997.198464,Poland,Unknown,89DA,2023-01-23,193
663353,DB2B68FE,RM,MTS,28,1003.007986,Poland,Unknown,7DD3,2025-07-28,190
128328,280700E7,FG,MTS,28,1000.212000,Poland,Unknown,00F9,2023-07-01,188
745041,913D91BB,FG,MTS,28,1000.216095,Poland,Unknown,AD85,2025-11-21,186
458226,6DF91D74,FG,MTS,28,1017.502972,United States,US-DA,612B,2024-10-10,183
654603,D15827BC,FG,MTS,28,994.709798,United States,US-F4,B17E,2025-07-16,182
406435,7A5D0A75,RM,MTS,28,1000.898738,United States,US-F4,B23B,2024-07-29,182
350973,3794A369,WIP,MTS,28,992.850634,United States,US-30,8552,2024-05-12,181
151549,6DF91D74,FG,MTS,28,1017.502972,Poland,Unknown,2C03,2023-08-03,181
121140,6B18007F,RM,MTS,28,990.860179,Japan,JP-4F,36FB,2023-06-21,181


We can also do a similar evaluation by total sales value. We would just need to remember to multiple the order quantity by the unit price in order to get that value.

In [54]:
trans_mts["sales_value"] = trans_mts["unit_price"] * trans_mts["order_quantity"]

top_sales = trans_mts.sort_values("sales_value", ascending=False).head(10)
bottom_sales = trans_mts.sort_values("sales_value", ascending=False).tail(10)

pd.concat([top_sales, bottom_sales])

,sku_number,inventory_type,stocking_type,lead_time,unit_price,country,manufacturing_site,division_code,transaction_date,order_quantity,sales_value
16266,01DD6D59,WIP,MTS,28,997.198464,Poland,Unknown,89DA,2023-01-23,193,192459.303625
663353,DB2B68FE,RM,MTS,28,1003.007986,Poland,Unknown,7DD3,2025-07-28,190,190571.517276
128328,280700E7,FG,MTS,28,1000.212000,Poland,Unknown,00F9,2023-07-01,188,188039.855906
458226,6DF91D74,FG,MTS,28,1017.502972,United States,US-DA,612B,2024-10-10,183,186203.043911
745041,913D91BB,FG,MTS,28,1000.216095,Poland,Unknown,AD85,2025-11-21,186,186040.193706
151549,6DF91D74,FG,MTS,28,1017.502972,Poland,Unknown,2C03,2023-08-03,181,184168.037967
700366,8481C986,FG,MTS,28,1018.063189,Japan,JP-4F,9667,2025-09-19,180,183251.373946
406435,7A5D0A75,RM,MTS,28,1000.898738,United States,US-F4,B23B,2024-07-29,182,182163.570253
654603,D15827BC,FG,MTS,28,994.709798,United States,US-F4,B17E,2025-07-16,182,181037.183221
527654,86F79768,RM,MTS,28,993.170017,United States,US-F4,B23B,2025-01-17,181,179763.773152


### Safety Stock
#### Aggregation
Now we can start on calculating the safety stock levels for each SKU. Before doing that, we have to aggregate the data by the SKUs. We'll look at the descriptive statistics for each SKU based on the order quantites as well as the average lead time.

To start, we group the data by SKUs and find the average of both the order quantity and lead time.

In [55]:
overview_a = (
    trans_mts
    .groupby("sku_number")
    .aggregate({"order_quantity": "mean",
                "lead_time": "mean"})
    .rename(columns={"order_quantity":"avg_quantity",
                     "lead_time": "avg_lead_time"})
    )
overview_a

,avg_quantity,avg_lead_time
sku_number,,
0053EE13,98.004491,27.756063
009B5641,9.680000,14.000000
00CF56D8,99.964731,28.000000
011D14D9,8.721739,14.000000
01DD6D59,100.991821,28.000000
...,...,...
FEECD992,100.357260,28.000000
FEF8112E,95.652197,27.341004
FF5D718B,92.527617,28.000000


From there, we need to find the min, max, median, variance, and standard deviation of the order quantity. To make it easier to find all these statistics, we make sure to select only the sku and quantity columns. Then with the modified table, we can find the rest of the order quantity statistics.

In [56]:
trans_mts_o = trans_mts[["sku_number", "order_quantity"]]
trans_mts_o

,sku_number,order_quantity
1,7F93DE39,103
2,F7353AF9,6
3,DF10CC32,2
5,AAFE4414,112
6,1D7F004E,80
...,...,...
772799,20F7AA6B,97
772800,FDC8BB25,14
772801,280700E7,97
772802,8D256BFE,89


In [57]:
overview_min = (
    trans_mts_o
    .groupby("sku_number")
    .min()
    .rename(columns={"order_quantity": "min_quantity"})
)

overview_max = (
    trans_mts_o
    .groupby("sku_number")
    .max()
    .rename(columns={"order_quantity": "max_quantity"})
)

overview_med = (
    trans_mts_o
    .groupby("sku_number")
    .median()
    .rename(columns={"order_quantity": "median_quantity"})
)

overview_var = (
    trans_mts_o
    .groupby("sku_number")
    .var()
    .rename(columns={"order_quantity": "variance_quantity"})
)

overview_sd = (
    trans_mts_o
    .groupby("sku_number")
    .std()
    .rename(columns={"order_quantity": "standard_deviation_quantity"})
)

With all the statistics found, we can combine them into one table.

In [58]:
mm = pd.merge(overview_min, overview_max, on="sku_number")
mmm = pd.merge(mm, overview_med, on="sku_number")
mmmv = pd.merge(mmm, overview_var, on="sku_number")
mmmvs = pd.merge(mmmv, overview_sd, on="sku_number")
overview = pd.merge(mmmvs, overview_a, on="sku_number")
overview

,min_quantity,max_quantity,median_quantity,variance_quantity,standard_deviation_quantity,avg_quantity,avg_lead_time
sku_number,,,,,,,
0053EE13,0,170,99.0,543.533038,23.313795,98.004491,27.756063
009B5641,0,41,8.0,57.472323,7.581050,9.680000,14.000000
00CF56D8,38,157,99.0,389.829917,19.744111,99.964731,28.000000
011D14D9,0,26,7.0,40.763997,6.384669,8.721739,14.000000
01DD6D59,22,193,101.0,411.857544,20.294274,100.991821,28.000000
...,...,...,...,...,...,...,...
FEECD992,23,170,100.0,393.999488,19.849420,100.357260,28.000000
FEF8112E,0,174,99.0,777.872167,27.890360,95.652197,27.341004
FF5D718B,0,152,98.0,953.538842,30.879424,92.527617,28.000000


#### Safety Stock - Normal

Assuming a normal distribution, safety stock levels can be calculated using the formula: 

#### SS = z<sub>α</sub> * σ<sub>D</sub> * √L

where **z<sub>α</sub>** is the z-score of the service level coefficient **α**, which is the probability of satisfying an order, **σ<sub>D</sub>** is the standard deviation of the demand, and **L** is the lead time.

We start by finding the z-score of the 75%, 90%, and 95% service level coefficients, storing the values in their respective `sl##` objects to prevent unnecessary rounding.

In [59]:
sl75 = stats.norm.ppf(0.75)
sl90 = stats.norm.ppf(0.90)
sl95 = stats.norm.ppf(0.95)

We can then fill in the safety stock formula with the repective columns in order to calculate the safety stock levels at each service level coefficient.

In [60]:
overview.insert(0, "0.75_ss", 
                sl75*overview["standard_deviation_quantity"]*(overview["avg_lead_time"]**0.5))
overview.insert(1, "0.90_ss", 
                sl90*overview["standard_deviation_quantity"]*(overview["avg_lead_time"]**0.5))
overview.insert(2, "0.95_ss", 
                sl95*overview["standard_deviation_quantity"]*(overview["avg_lead_time"]**0.5))

overview

,0.75_ss,0.90_ss,0.95_ss,min_quantity,max_quantity,median_quantity,variance_quantity,standard_deviation_quantity,avg_quantity,avg_lead_time
sku_number,,,,,,,,,,
0053EE13,82.845182,157.408430,202.031533,0,170,99.0,543.533038,23.313795,98.004491,27.756063
009B5641,19.132369,36.352098,46.657413,0,41,8.0,57.472323,7.581050,9.680000,14.000000
00CF56D8,70.468001,133.891400,171.847752,38,157,99.0,389.829917,19.744111,99.964731,28.000000
011D14D9,16.113050,30.615297,39.294309,0,26,7.0,40.763997,6.384669,8.721739,14.000000
01DD6D59,72.431567,137.622237,176.636229,22,193,101.0,411.857544,20.294274,100.991821,28.000000
...,...,...,...,...,...,...,...,...,...,...
FEECD992,70.843857,134.605539,172.764338,23,170,100.0,393.999488,19.849420,100.357260,28.000000
FEF8112E,98.364118,186.894892,239.876997,0,174,99.0,777.872167,27.890360,95.652197,27.341004
FF5D718B,110.210650,209.403673,268.766705,0,152,98.0,953.538842,30.879424,92.527617,28.000000


With the safety stock level calculated, we can convert the resulting quantities to whole numbers to better reflect inventory amounts.

In [61]:
safetystock_norm = overview[["0.75_ss", "0.90_ss", "0.95_ss"]].astype("int64")
safetystock_norm

,0.75_ss,0.90_ss,0.95_ss
sku_number,,,
0053EE13,82,157,202
009B5641,19,36,46
00CF56D8,70,133,171
011D14D9,16,30,39
01DD6D59,72,137,176
...,...,...,...
FEECD992,70,134,172
FEF8112E,98,186,239
FF5D718B,110,209,268


This allows us to see the SKU distribution by the service level coefficients. For instance, at the 95% coefficient:

The largest safety stock

In [62]:
safetystock_norm.sort_values("0.95_ss", ascending=False).head(1)

,0.75_ss,0.90_ss,0.95_ss
sku_number,,,
7D640DC9,148,281,361


The smallest safety stock

In [63]:
safetystock_norm.sort_values("0.95_ss").head(1)

,0.75_ss,0.90_ss,0.95_ss
sku_number,,,
16A11D4A,12,24,31


The average safety stock level across all SKUs

In [64]:
int(safetystock_norm["0.95_ss"].mean())

118

#### Safety Stock - Non-Quantile
Though normal distribution is assumed here, it isn't always the case for demand. One way we can compensate for that is to use empirical quantiles of each SKU and calculate their respective z-scores from there.

We start by finding the empirical quantile valules of each SKU at the 75th, 90th, and 95th percentiles, combining them into one table.

In [65]:
quan75 = (trans_mts
    .groupby("sku_number")
    .aggregate({"order_quantity": lambda x: stats.mstats.mquantiles(x, prob=0.75)[0]})
    .rename(columns={"order_quantity": "0.75_quan"})
    )

quan90 = (trans_mts
    .groupby("sku_number")
    .aggregate({"order_quantity": lambda x: stats.mstats.mquantiles(x, prob=0.90)[0]})
    .rename(columns={"order_quantity": "0.90_quan"})
    )

quan95 = (trans_mts
    .groupby("sku_number")
    .aggregate({"order_quantity": lambda x: stats.mstats.mquantiles(x, prob=0.95)[0]})
    .rename(columns={"order_quantity": "0.95_quan"})
    )

q = pd.merge(quan75, quan90, on="sku_number")
quantiles = pd.merge(q, quan95, on="sku_number")
quantiles

,0.75_quan,0.90_quan,0.95_quan
sku_number,,,
0053EE13,113.0,125.88,133.00
009B5641,13.0,21.58,24.59
00CF56D8,114.0,126.00,132.00
011D14D9,13.0,18.00,19.84
01DD6D59,115.0,127.00,134.89
...,...,...,...
FEECD992,114.0,126.00,132.34
FEF8112E,112.0,126.00,133.00
FF5D718B,112.0,124.00,132.00


This table is then merged into the overview table so that all the information is all in one place.

In [66]:
over_quan = pd.merge(overview, quantiles, on="sku_number")
over_quan

,0.75_ss,0.90_ss,0.95_ss,min_quantity,max_quantity,median_quantity,variance_quantity,standard_deviation_quantity,avg_quantity,avg_lead_time,0.75_quan,0.90_quan,0.95_quan
sku_number,,,,,,,,,,,,,
0053EE13,82.845182,157.408430,202.031533,0,170,99.0,543.533038,23.313795,98.004491,27.756063,113.0,125.88,133.00
009B5641,19.132369,36.352098,46.657413,0,41,8.0,57.472323,7.581050,9.680000,14.000000,13.0,21.58,24.59
00CF56D8,70.468001,133.891400,171.847752,38,157,99.0,389.829917,19.744111,99.964731,28.000000,114.0,126.00,132.00
011D14D9,16.113050,30.615297,39.294309,0,26,7.0,40.763997,6.384669,8.721739,14.000000,13.0,18.00,19.84
01DD6D59,72.431567,137.622237,176.636229,22,193,101.0,411.857544,20.294274,100.991821,28.000000,115.0,127.00,134.89
...,...,...,...,...,...,...,...,...,...,...,...,...,...
FEECD992,70.843857,134.605539,172.764338,23,170,100.0,393.999488,19.849420,100.357260,28.000000,114.0,126.00,132.34
FEF8112E,98.364118,186.894892,239.876997,0,174,99.0,777.872167,27.890360,95.652197,27.341004,112.0,126.00,133.00
FF5D718B,110.210650,209.403673,268.766705,0,152,98.0,953.538842,30.879424,92.527617,28.000000,112.0,124.00,132.00


With everything in one table, we can now calculate the safety stock level using the empirical quantile values. Instead of using the normal distribution z-score, we can calulate the z-score with the formula:
#### z = (x-μ)/σ
where x is the quantile value we found, μ is the mean, and σ is the standard deviation.

In [67]:
zq75 = (over_quan["0.75_quan"]-over_quan["avg_quantity"])/over_quan["standard_deviation_quantity"]
zq90 = (over_quan["0.90_quan"]-over_quan["avg_quantity"])/over_quan["standard_deviation_quantity"]
zq95 = (over_quan["0.95_quan"]-over_quan["avg_quantity"])/over_quan["standard_deviation_quantity"]

over_quan.insert(3, "0.75_ssq", 
                zq75*over_quan["standard_deviation_quantity"]*(over_quan["avg_lead_time"]**0.5))
over_quan.insert(4, "0.90_ssq", 
                zq90*over_quan["standard_deviation_quantity"]*(over_quan["avg_lead_time"]**0.5))
over_quan.insert(5, "0.95_ssq", 
                zq95*over_quan["standard_deviation_quantity"]*(over_quan["avg_lead_time"]**0.5))

over_quan.loc[:, "0.75_ss":"0.95_ssq"]

,0.75_ss,0.90_ss,0.95_ss,0.75_ssq,0.90_ssq,0.95_ssq
sku_number,,,,,,
0053EE13,82.845182,157.408430,202.031533,79.002375,146.859396,184.370420
009B5641,19.132369,36.352098,46.657413,12.422303,44.525723,55.788112
00CF56D8,70.468001,133.891400,171.847752,74.267661,137.765692,169.514708
011D14D9,16.113050,30.615297,39.294309,16.007786,34.716073,41.600723
01DD6D59,72.431567,137.622237,176.636229,74.124315,137.622347,179.372302
...,...,...,...,...,...,...
FEECD992,70.843857,134.605539,172.764338,72.190593,135.688624,169.236751
FEF8112E,98.364118,186.894892,239.876997,85.480418,158.684494,195.286532
FF5D718B,110.210650,209.403673,268.766705,103.038165,166.536197,208.868218


We can then convert the resulting calculations to get each SKUs safety stock value based on quantile calculations.

In [68]:
safetystock_quan = over_quan.loc[:, "0.75_ssq":"0.95_ssq"].astype(int)
safetystock_quan

,0.75_ssq,0.90_ssq,0.95_ssq
sku_number,,,
0053EE13,79,146,184
009B5641,12,44,55
00CF56D8,74,137,169
011D14D9,16,34,41
01DD6D59,74,137,179
...,...,...,...
FEECD992,72,135,169
FEF8112E,85,158,195
FF5D718B,103,166,208
